In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from collections import Counter
import random
from PIL import Image

# =================================================================
# 🧁 데이터셋 정보: Food-101 (음식 이미지 분류)
# 설명: 이 데이터셋은 101가지의 다양한 음식 이미지를 포함하고 있습니다.
# 목표: 주어진 이미지가 어떤 종류의 음식인지 분류(Image Classification)하는 것이 목표입니다.
# 학습 목표: 파이썬을 사용해 대규모 이미지 데이터셋을 불러오고, 탐색하고, 간단히 분석하는 방법을 익힙니다!
# =================================================================

# --- 설정 값 ---
DATASET_NAME = "ethz/food101"
SPLIT_NAME = "train"
SAMPLE_COUNT = 10  # 실습 시 상위 10개 샘플만 사용합니다.
# -----------------


def load_dataset_with_fallback(name, split, sample_count):
    """
    스트리밍 모드(streaming=True)로 데이터셋을 로드하고, 실패할 경우 일반 모드로 전환합니다.
    """
    print("✨ [1단계] 데이터 로딩 시도: 스트리밍 모드 (streaming=True)로 시작합니다.")
    try:
        # 📌 핵심 패턴: 스트리밍 모드로 데이터셋 로드
        dataset = load_dataset(name, split=split, streaming=True)
        print("✅ 스트리밍 모드 로딩 성공! 데이터 탐색 준비 완료.")
        return dataset
    except Exception as e:
        print(f"\n⚠️ [오류 발생] 스트리밍 모드 로딩 중 오류가 발생했습니다: {e}")
        print("👉 일반 로딩 모드(streaming=False)로 전환하여 재시도합니다...")
        try:
            # 📌 대체 패턴: 일반 모드로 로드
            return load_dataset(name, split=split, streaming=False)
        except Exception as e_fail:
            print(f"❌ 치명적인 오류: 일반 모드 로딩에도 실패했습니다. ({e_fail})")
            return None

def run_food_classifier_tutorial():
    """
    Food-101 데이터셋을 활용한 초급 AI 실습 과정을 순서대로 진행합니다.
    """
    global dataset_iterator
    dataset = load_dataset_with_fallback(DATASET_NAME, SPLIT_NAME, SAMPLE_COUNT)

    if dataset is None:
        print("\n😢 데이터 로드에 실패하여 실습을 진행할 수 없습니다. 코드를 확인해주세요.")
        return

    print("\n" + "="*60)
    print("🧠 Food-101 데이터 탐색 및 분석 실습 시작! 🤖")
    print("="*60)

    # =============================================================
    # Part 1: 데이터 구조 및 레이블 이해하기 (탐색 단계)
    # =============================================================
    print("\n--- [🌟 Part 1] 데이터셋 구조 파악하기: 이 데이터가 무엇으로 이루어져 있을까? ---")
    
    # 💡 constraint 해결: streaming 모드에 맞는 iterator 패턴 사용
    dataset_iterator = iter(dataset.take(SAMPLE_COUNT))

    # 첫 번째 샘플을 가져와서 구조를 확인합니다.
    try:
        sample_data = next(dataset_iterator)
        print(f"🔎 샘플 데이터 구조 (첫 번째 샘플): {sample_data}")
        print(f"   - Key 1: 'image' (이미지 데이터가 저장된 위치)")
        print(f"   - Key 2: 'label' (음식의 이름, 문자열)")
    except StopIteration:
        print("데이터셋이 비어 있습니다. 샘플을 확인할 수 없습니다.")
        return
    
    # 🍴 Label Mapping: 레이블 이름을 숫자로 변환하는 과정 이해하기
    # Food-101은 문자열 레이블을 가지고 있으므로, 분석을 위해 딕셔너리 형태로 매핑해주는 것이 좋습니다.
    print("\n🔗 [TIP] 레이블 이름 -> 숫자 인덱스 매핑을 해보겠습니다!")
    sample_data_list = list(dataset.take(1)) # Config 자체를 로드하여 feature 정보를 확인
    if sample_data_list:
        # 실제 데이터셋 구조를 파싱하는 대신, feature 정보를 활용합니다.
        # datasets 라이브러리가 내부적으로 가지고 있는 정보를 이용합니다.
        from datasets import load_dataset_builder
        try:
             builder = load_dataset_builder(DATASET_NAME)
             feature = builder.info.features['label']
             class_names = feature.names
             print(f"   => 총 클래스 개수: {len(class_names)}가지입니다. (엄청나게 다양하죠?)")
             print(f"   => 예시 클래스 이름: {class_names[0]}, {class_names[1]}, ..., {class_names[-1]}")
        except Exception as e:
            print(f"  (경고: 클래스 정보 로드 실패, {e})")
    
    # =============================================================
    # Part 2: 데이터 분석 및 통계 (분석가처럼 생각하기)
    # =============================================================
    print("\n" + "="*60)
    print("📊 [🌟 Part 2] 데이터 분석: 우리 주변에서 가장 많이 발견되는 음식은 무엇일까? (Top 5 카운트)")
    print("="*60)
    
    # 🧺 빈도수 계산기 준비
    label_counts = []
    
    print(f"🚀 상위 {min(SAMPLE_COUNT, 20)}개의 샘플을 순회하며 레이블 빈도수를 계산합니다...")

    # Iterator를 사용하여 데이터를 한 개씩 처리하며 레이블을 수집합니다.
    # streaming 모드에서 반복하는 표준 패턴입니다.
    for i, sample in enumerate(dataset):
        if i >= SAMPLE_COUNT * 2: # 무한 루프 방지 및 예시를 위해 제한
             break
        try:
            label_name = sample['label']
            label_counts.append(label_name)
        except Exception as e:
            # 간혹 데이터셋 로드 시 키 이름 오류가 발생할 수 있습니다.
            print(f"⚠️ {i+1}번째 샘플에서 레이블을 읽는 오류가 발생했습니다: {e}")
            continue
    
    # 📊 빈도수 계산 및 시각화
    food_counts = Counter(label_counts)
    top_foods = food_counts.most_common(5)

    print("\n⭐ 데이터 분석 결과: 가장 흔하게 발견되는 상위 5가지 음식!")
    for food, count in top_foods:
        print(f"   - {food}: {count}번 (오, 이 음식들이 가장 많군요!)")
        
    # 🖼️ Visualization (Matplotlib 사용)
    plt.figure(figsize=(10, 5))
    plt.bar(
        [f.replace('_', ' ').title() for f, c in top_foods], # 출력용 이름 변환
        [c for f, c in top_foods],
        color='skyblue'
    )
    plt.xlabel("Food Item (Top 5)")
    plt.ylabel("Count (빈도수)")
    plt.title("Top 5 Food Items in the Dataset")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


    # =============================================================
    # Part 3: 이미지 전처리 및 AI 개념 실습 (실전 감각 익히기)
    # =============================================================
    print("\n" + "="*60)
    print("📸 [🌟 Part 3] 이미지 전처리 및 예측 시뮬레이션: 이미지를 화면에 띄워보자!")
    print("="*60)
    
    print("🚨 주의: 이미지 데이터는 메모리를 많이 사용합니다. 대표적인 3개만 살펴보겠습니다.")

    # 💡 Image Handling: 이미지는 PIL 객체이므로, Matplotlib로 보여줘야 합니다.
    plt.figure(figsize=(15, 5))
    
    # 3개의 샘플만 테스트합니다.
    sample_display_count = 0
    
    # 재순회하거나, 다시 처음부터 가져옵니다. (간단한 시연을 위해 반복)
    for i, sample in enumerate(dataset):
        if sample_display_count >= 3:
            break
        
        try:
            # PIL Image 객체를 가져옵니다.
            image = sample['image']
            label = sample['label']

            print(f"   [{sample_display_count + 1}/{3}] 분석 중: {label} (이미지 로드 성공)")
            
            # Matplotlib을 사용하여 이미지 출력 (PIL 이미지는 직접 show()로 띄우지 않습니다)
            plt.subplot(1, 3, sample_display_count + 1)
            plt.imshow(image)
            plt.title(f"Food: {label.replace('_', ' ').title()}")
            plt.axis('off')
            
            sample_display_count += 1
        except Exception as e:
            print(f"❌ 샘플 {i} 처리 중 이미지 오류 발생: {e}")
            pass

    plt.suptitle("Random Food Samples from Food-101", fontsize=16)
    plt.show()
    
    print("\n=================================================================")
    print("🎉 실습 완료! 축하드립니다! 🎉")
    print("축하해요! 여러분은 거대한 이미지 분류 데이터셋을 성공적으로 불러오고, ")
    print("분석하고, 시각적으로 탐색하는 AI의 기본적인 단계를 마쳤습니다!")
    print("이후에는 모델 학습(Model Training)을 통해 진짜 예측을 해낼 수 있답니다. 😉")
    print("=================================================================")


if __name__ == "__main__":
    run_food_classifier_tutorial()